# 🛫 ADAPT ATC — GRPO Training | T4 GPU (fp16)

**Multi-Agent Air Traffic Control with GRPO + Qwen2.5-1.5B-Instruct**

| Setting | Value |
|---------|-------|
| GPU | T4 16 GB |
| dtype | **fp16** (T4 = Turing, no native bf16) |
| Model | Qwen2.5-1.5B-Instruct 4-bit QLoRA rank=8 |
| Batch | 1 × accum 4 = eff. 4 |
| Generations | 2 per prompt |
| Episodes | 150 (~2 h) |
| Reward | Format → Safety → Quality (composable rubric) |

> **One-click**: Runtime → Change runtime type → T4 GPU → Run All

## What trains
Three agents share one LLM (system-prompt-differentiated):
- **AMAN** – arrival sequencing (wake turbulence, emergency priority)
- **DMAN** – departure sequencing (ATFM deadlines, emergency handling)
- **ADAPT** – domain transfer (maps ICU/port signals → ATC parameters)


## Step 1 — Install dependencies

In [ ]:
# Install in one shot — unsloth handles torch/cuda compatibility
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

# Unsloth first (manages torch version)
pip("unsloth")
# Then TRL + ecosystem
pip("trl>=0.15", "datasets", "bitsandbytes", "accelerate", "peft", "matplotlib")

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ Dependencies installed")

## Step 2 — GPU check + dtype selection

In [ ]:
import torch

assert torch.cuda.is_available(), "❌ No GPU! Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

# T4 = Turing architecture → no native bf16 matmul → must use fp16
# A100/H100 = Ampere → use bf16 (see train_a100.ipynb)
IS_BF16 = torch.cuda.is_bf16_supported()
DTYPE   = torch.bfloat16 if IS_BF16 else torch.float16
print(f"dtype: {'bfloat16 (Ampere+)' if IS_BF16 else 'float16 (T4 Turing)'}")

if IS_BF16:
    print("ℹ️  bf16 detected — this notebook is optimised for T4 (fp16).")
    print("   For A100/H100 use train_a100.ipynb for larger batch & more generations.")

## Step 3 — Repository setup

**Option A** (recommended): Upload `adapt-atc-final.zip` via Colab Files panel then run the unzip cell.

**Option B**: Mount Google Drive if the folder is already there.

**Option C**: Clone from GitHub if you pushed it there.

In [ ]:
import os, sys, shutil

REPO_PATH = "/content/adapt-atc-final"

# ── Option A: unzip uploaded file ────────────────────────────────────────────
# If you uploaded adapt-atc-final.zip, uncomment:
# !unzip -q /content/adapt-atc-final.zip -d /content/

# ── Option B: Google Drive ────────────────────────────────────────────────────
# Uncomment if your repo is at MyDrive/adapt-atc-final:
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE = "/content/drive/MyDrive/adapt-atc-final"
# if os.path.exists(DRIVE) and not os.path.exists(REPO_PATH):
#     shutil.copytree(DRIVE, REPO_PATH)

# ── Option C: GitHub clone ────────────────────────────────────────────────────
# Uncomment and replace URL:
# if not os.path.exists(REPO_PATH):
#     !git clone https://github.com/YOUR_USERNAME/adapt-atc-final.git /content/adapt-atc-final

# ── Verify ───────────────────────────────────────────────────────────────────
if not os.path.exists(REPO_PATH):
    raise FileNotFoundError(
        f"Repo not found at {REPO_PATH}.\n"
        "Upload adapt-atc-final.zip → Colab Files → then uncomment Option A above."
    )

sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)
print(f"✅ Repo ready: {REPO_PATH}")
print(f"   Files: {sorted(os.listdir(REPO_PATH))[:10]}")

## Step 4 — Import repo modules

In [ ]:
try:
    from tasks import task_catalog
    from training.reward_functions import aman_reward_fn, dman_reward_fn, adapt_reward_fn
    from training.dataset import build_episode_dataset, AMAN_SYSTEM, DMAN_SYSTEM
    from multi_agent.models import AgentRole
    print("✅ All imports OK")
    catalog = task_catalog()
    print(f"   Tasks: {list(catalog.keys())}")
except Exception as e:
    print(f"❌ Import error: {e}")
    print("Ensure REPO_PATH in Step 3 is correct and all repo files are present.")
    raise

## Step 5 — Load model (fp16, 4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL

# CRITICAL: Patch BEFORE loading the model — sets up GRPO generation hooks
try:
    PatchFastRL("GRPO", FastLanguageModel)
    print("✅ PatchFastRL applied")
except Exception as e:
    print(f"ℹ️  PatchFastRL: {e}  (newer Unsloth may not need this — continuing)")

# T4 settings: 4-bit quant + fp16 = ~1 GB model weight
# Leaving ~15 GB for activations, gradients, and batch
MODEL_NAME  = "unsloth/Qwen2.5-1.5B-Instruct"  # pre-quantized mirror
MAX_SEQ_LEN = 2048   # prompt 1536 + completion 256 = 1792 < 2048

print(f"Loading {MODEL_NAME} in 4-bit {'fp16' if not IS_BF16 else 'bf16'} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,           # auto: fp16 on T4, bf16 on A100 — never float32
    trust_remote_code=True,
)

# Pad token required by GRPO's left-padded batching
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

total = sum(p.numel() for p in model.parameters())
dtype = next(model.parameters()).dtype
print(f"✅ Loaded: {total/1e6:.0f} M params | dtype={dtype}")

## Step 6 — Apply LoRA (rank=8)

In [ ]:
LORA_RANK = 8  # ~0.4% trainable params for 1.5B — fast convergence

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,   # standard: alpha = 2 × rank
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Unsloth's memory-efficient variant
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA rank={LORA_RANK}  |  {trainable:,} / {total:,} trainable ({100*trainable/total:.2f}%)")

## Step 7 — Build training dataset

In [ ]:
from datasets import Dataset
from collections import Counter
import time

N_EPISODES = 150  # ~2 hours on T4 — increase to 200 for better results if time allows

print(f"Building {N_EPISODES}-episode dataset (ADAPT:65% / AMAN:17% / DMAN:18%) ...")
t0 = time.time()

dataset_raw = build_episode_dataset(
    n_episodes=N_EPISODES,
    seed=42,
    include_adapt=True,
    domain_episode_ratio=0.65,   # ADAPT is the primary training role
    long_horizon_ratio=0.0,      # keep off for T4 memory budget
)
dataset = Dataset.from_list(dataset_raw)

role_counts = Counter(s["agent_role"] for s in dataset_raw)
print(f"✅ {len(dataset)} samples in {time.time()-t0:.1f}s")
for role, cnt in sorted(role_counts.items()):
    print(f"   {role:8s}: {cnt} samples")

## Step 8 — Reward function dispatcher

Routes each completion to its role-specific composable rubric.
Always returns plain Python `float` — never a tensor.

In [ ]:
import re as _re, json as _json

_DISPATCH = {
    "AMAN":  aman_reward_fn,
    "DMAN":  dman_reward_fn,
    "ADAPT": adapt_reward_fn,
}


def _partial_credit(text: str, role: str) -> float:
    """Variable partial credit when JSON parsing fully fails.
    Keeps reward_std > 0 so GRPO advantage is non-zero during cold start.
    """
    s = 0.05
    if "{" in text: s += 0.03
    if role == "AMAN"  and "arrival_slots"   in text: s += 0.06
    if role == "DMAN"  and "departure_slots" in text: s += 0.06
    if role == "ADAPT" and "entity_wake_map" in text: s += 0.06
    if '"flight_id"'  in text: s += 0.04
    if "rationale"    in text: s += 0.02
    return min(s, 0.22)  # cap: full rubric (0.4-0.9) must dominate once JSON learned


def reward_fn(completions, **kwargs):
    """Multi-agent ATC reward dispatcher.
    
    TRL GRPOTrainer calls this after generating `num_generations` completions per prompt.
    Returns: List[float] guaranteed — no tensors, no NaN.
    """
    n     = len(completions)
    roles = kwargs.get("agent_role", ["AMAN"] * n)
    if not isinstance(roles, list):
        roles = [roles] * n
    while len(roles) < n:
        roles.append(roles[-1] if roles else "AMAN")

    rewards = []
    for i, (comp, role) in enumerate(zip(completions, roles)):
        # TRL may pass completion as list-of-dicts (conversational format)
        if isinstance(comp, list):
            comp = comp[-1].get("content", "") if comp else ""
        comp = str(comp)

        fn = _DISPATCH.get(str(role), aman_reward_fn)

        # Build per-sample kwargs slice (reward fn expects lists of length 1)
        sample_kw = {}
        for k, v in kwargs.items():
            if k == "agent_role":
                continue
            if isinstance(v, list):
                sample_kw[k] = [v[i] if i < len(v) else (v[-1] if v else "")]
            else:
                sample_kw[k] = [v]

        try:
            r      = fn([comp], **sample_kw)
            reward = float(r[0]) if r else _partial_credit(comp, str(role))
            reward = max(0.0, min(1.0, reward))
            if reward != reward:   # NaN guard
                reward = _partial_credit(comp, str(role))
        except Exception:
            reward = _partial_credit(comp, str(role))

        rewards.append(float(reward))

    return rewards


# ── Sanity check ──────────────────────────────────────────────────────────────
test_r = reward_fn(
    ["{", "{\"arrival_slots\":[{\"flight_id\":\"AI101\",\"runway\":\"28R\",\"assigned_minute\":10,\"hold_minutes\":0}],\"rationale\":\"emergency first\"}"],
    task_id=["delhi_monsoon_recovery_easy", "delhi_monsoon_recovery_easy"],
    agent_role=["AMAN", "AMAN"],
)
assert all(isinstance(r, float) for r in test_r), "Reward must be float!"
print(f"✅ Reward fn OK | partial={test_r[0]:.3f}  full-parse={test_r[1]:.3f}")

## Step 9 — Configure GRPO

**T4 key settings:**
- `fp16=True` — explicit, never auto-detect (prevents dtype mismatch)
- `num_generations=2` — minimum for stable GRPO advantage variance
- `optim="adamw_8bit"` — NOT `paged_adamw_8bit` (fails on some bitsandbytes versions)
- `beta=0.0` — KL=0 avoids `ref_per_token_logps=None` crash with PEFT

In [ ]:
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./adapt-atc-t4-out"

training_args = GRPOConfig(
    # ── Generation ────────────────────────────────────────────
    num_generations=2,           # T4 max: 2 (memory limit)
    max_completion_length=256,   # 256 tokens: enough for JSON + rationale
    max_prompt_length=1536,      # 1536 + 256 = 1792 < MAX_SEQ_LEN=2048

    # ── Optimisation ──────────────────────────────────────────
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,   # effective batch = 4
    num_train_epochs=3,
    warmup_ratio=0.10,
    max_grad_norm=1.0,

    # ── Precision — EXPLICIT for T4 (never rely on auto-detect) ──
    fp16=True,     # T4 Turing = fp16
    bf16=False,    # bf16 off — T4 has no Tensor Core bf16 matmul

    # ── Memory ────────────────────────────────────────────────
    optim="adamw_8bit",       # bitsandbytes 8-bit Adam — stable on all bnb versions
    gradient_checkpointing=True,

    # ── KL regularisation ─────────────────────────────────────
    beta=0.0,   # 0 = no KL; avoids ref_per_token_logps=None with PEFT

    # ── Logging / saving ──────────────────────────────────────
    output_dir=OUTPUT_DIR,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    seed=42,
    run_name="adapt-atc-grpo-t4",
)

print("✅ GRPOConfig ready")
print(f"   precision     : fp16")
print(f"   batch         : {training_args.per_device_train_batch_size} × accum {training_args.gradient_accumulation_steps}")
print(f"   generations   : {training_args.num_generations} per prompt")
print(f"   completion len: {training_args.max_completion_length} tokens")
print(f"   KL coeff      : {training_args.beta}")
print(f"   output        : {OUTPUT_DIR}")

## Step 10 — Train!

Reward should increase from ~0.05–0.15 (cold start) to ~0.4–0.7 over training.
If reward stays flat, check that `reward_std > 0` — if std=0, generations are identical.

In [ ]:
import time, os

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_fn],
    args=training_args,
    train_dataset=dataset,
)

print("🚀 GRPO training started!")
print("   Watch for reward_mean increasing in the log below.")
print(f"   Estimated time: ~{N_EPISODES*50//60}–{N_EPISODES*70//60} min on T4\n")

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f"\n✅ Training done in {elapsed/60:.0f} min")

## Step 11 — Save model

In [ ]:
import json

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Save training log for analysis
log_path = f"{OUTPUT_DIR}/log_history.json"
with open(log_path, "w") as f:
    json.dump(trainer.state.log_history, f, indent=2, default=str)

print(f"✅ Model saved : {OUTPUT_DIR}")
print(f"   Log         : {log_path}")

## Step 12 — Reward curve plot

In [ ]:
import matplotlib.pyplot as plt, json, os

log = trainer.state.log_history

# Collect reward values — TRL version-agnostic key search
reward_vals = []
for entry in log:
    for k in ["rewards/reward_fn", "reward", "train/reward",
              "rewards/combined_reward_fn", "rewards/reward_fn_mean"]:
        if k in entry and isinstance(entry[k], (int, float)):
            reward_vals.append(float(entry[k]))
            break

if not reward_vals:
    # Fallback: any numeric key with 'reward' in name
    for entry in log:
        for k, v in entry.items():
            if "reward" in k.lower() and isinstance(v, (int, float)):
                reward_vals.append(float(v))
                break

if reward_vals:
    W = max(5, len(reward_vals) // 15)
    ma = [sum(reward_vals[max(0,i-W):i+1]) / min(i+1, W) for i in range(len(reward_vals))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(reward_vals, alpha=0.4, color="steelblue", label="step reward")
    axes[0].plot(ma, color="crimson", linewidth=2.5, label=f"MA-{W}")
    axes[0].set_xlabel("Training step"); axes[0].set_ylabel("Reward")
    axes[0].set_title("ADAPT ATC — Reward Curve (T4 fp16)"); axes[0].legend()
    axes[0].grid(True, alpha=0.25)

    q = max(1, len(reward_vals) // 4)
    first_q_mean = sum(reward_vals[:q]) / q
    last_q_mean  = sum(reward_vals[-q:]) / q
    axes[1].bar(["First 25%", "Last 25%"], [first_q_mean, last_q_mean],
                color=["salmon", "lightgreen"], edgecolor="black")
    for i, v in enumerate([first_q_mean, last_q_mean]):
        axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontweight="bold")
    axes[1].set_ylabel("Mean Reward"); axes[1].set_title("Training Improvement")
    axes[1].set_ylim(0, max(last_q_mean * 1.3, 0.2))

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/reward_curve_t4.png", dpi=150, bbox_inches="tight")
    plt.show()

    improvement = last_q_mean - first_q_mean
    arrow = "↑" if improvement > 0.02 else ("↓" if improvement < -0.02 else "→")
    print(f"\n📊 Reward summary:")
    print(f"   First 25% mean : {first_q_mean:.4f}")
    print(f"   Last 25% mean  : {last_q_mean:.4f}")
    print(f"   Improvement    : {improvement:+.4f} {arrow}")
else:
    print("⚠️  No reward values found in log_history.")
    avail = list(log[0].keys()) if log else []
    print(f"   Available keys: {avail}")

## Step 13 — Inference demo

Test the trained model on one arrival scheduling task.

In [ ]:
import torch, json, re

model.eval()


def infer(role: str = "AMAN", task_id: str = "delhi_monsoon_recovery_easy",
          max_new: int = 256, temperature: float = 0.3) -> str:
    from tasks import task_catalog
    task = task_catalog()[task_id]
    system = AMAN_SYSTEM if role == "AMAN" else DMAN_SYSTEM

    flights_text = "\n".join(
        f"  {f.flight_id}: {f.operation.value} {f.wake_class.value} "
        f"sched={f.scheduled_minute}min [{f.earliest_minute}-{f.latest_minute}] "
        f"runways={f.allowed_runways} priority={f.priority.value}"
        for f in task.flights[:12]
    )
    user = (
        f"Task: {task_id}\nRunways: {[r.runway_id for r in task.runways]}\n"
        f"Flights:\n{flights_text}\n\nProduce the optimal {role} plan."
    )

    messages = [{"role": "system", "content": system},
                {"role": "user",   "content": user}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=1536
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=temperature,
            do_sample=temperature > 0.01,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


print("=" * 60)
print("AMAN inference — Delhi Monsoon Recovery")
print("=" * 60)
response = infer("AMAN", "delhi_monsoon_recovery_easy")
print(response[:800])

# Parse JSON
clean = re.sub(r"```(?:json)?\s*|\s*```", "", response).strip()
try:
    data = json.loads(clean)
    slots = data.get("arrival_slots", [])
    print(f"\n✅ Valid JSON — {len(slots)} arrival slots")
    for s in slots[:5]:
        print(f"   {s.get('flight_id','?'):8s} → {s.get('runway','?'):4s} @ min {s.get('assigned_minute','?')}")
except json.JSONDecodeError:
    print("\nℹ️  Partial JSON (normal early in training — reward drives improvement)")

## Step 14 — Quick multi-agent eval (heuristic baseline vs trained)

In [ ]:
from multi_agent.inference import run_episode
from multi_agent.environment import MultiAgentATCEnvironment

tasks = ["delhi_monsoon_recovery_easy", "bengaluru_irrops_hard"]
env   = MultiAgentATCEnvironment(seed=99)

print("Heuristic baseline (no LLM):")
for tid in tasks:
    try:
        r = run_episode(task_id=tid, client=None, env=env, episode_id=0)
        print(f"  {tid:35s}  composite={r.get('composite',0):.3f}  "
              f"aman={r.get('aman_reward',0):.3f}  dman={r.get('dman_reward',0):.3f}")
    except Exception as e:
        print(f"  {tid}: {e}")

print("\n✅ For trained model eval, use train_grpo.py --eval_only --model ./adapt-atc-t4-out")